# YOLO용 데이터셋 만들기
여기서는 앞서 전처리한 COCO 형식의 JSON 파일들을 활용하여 YOLO 학습용 데이터를 만들고, 실제 학습을 하여 결과를 확인한다.</br>
그 첫 단계로, 우선 기본 dataset과 추가 dataset의 annotation 파일들을 각각 합쳐준다.

In [24]:
import os
import json
import shutil
import copy
from sklearn.model_selection import train_test_split
from pycocotools.coco import COCO
import pandas as pd
from PIL import Image
from glob import glob
from tqdm import tqdm
from ultralytics import YOLO

In [15]:
output_all_coco = {
    "images": [],
    "annotations": [],
    "categories": []
}
base_path='additional_training_data'
annotation_root='train_annotations'
folders=sorted(os.listdir(os.path.join(base_path,annotation_root)))
for folder in folders:
    files=copy.deepcopy([entry.name for entry in os.scandir(os.path.join(base_path,annotation_root,folder)) if entry.is_file() and entry.name.endswith(".json")])
    for file in files:
        with open(os.path.join(base_path,annotation_root,folder,file), "r", encoding="utf-8") as f:
            data=json.load(f)
            for image in data['images']:
                output_all_coco['images'].append(image)
            for annot in data['annotations']:
                output_all_coco['annotations'].append(annot)
            for cat in data['categories']:
                output_all_coco['categories'].append(cat)

t=pd.DataFrame(output_all_coco['categories'])
t=t.drop_duplicates()
output_all_coco['categories']=t.to_dict(orient='records')

save_path = os.path.join(base_path, "train.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output_all_coco, f, ensure_ascii=False, indent=4)

print("train.json 생성됨:", save_path)

train.json 생성됨: additional_training_data/train.json


In [17]:
output_all_coco = {
    "images": [],
    "annotations": [],
    "categories": []
}
base_path='ai06-level1-project'
annotation_root='train_annotations'
folders=sorted(os.listdir(os.path.join(base_path,annotation_root)))
for folder in folders:
    files=copy.deepcopy([entry.name for entry in os.scandir(os.path.join(base_path,annotation_root,folder)) if entry.is_file() and entry.name.endswith(".json")])
    for file in files:
        with open(os.path.join(base_path,annotation_root,folder,file), "r", encoding="utf-8") as f:
            data=json.load(f)
            for image in data['images']:
                output_all_coco['images'].append(image)
            for annot in data['annotations']:
                output_all_coco['annotations'].append(annot)
            for cat in data['categories']:
                output_all_coco['categories'].append(cat)

t=pd.DataFrame(output_all_coco['categories'])
t=t.drop_duplicates()
output_all_coco['categories']=t.to_dict(orient='records')

save_path = os.path.join(base_path, "train.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output_all_coco, f, ensure_ascii=False, indent=4)

print("train.json 생성됨:", save_path)

train.json 생성됨: ai06-level1-project/train.json


이제 두 파일을 합쳐준다.

In [19]:
#YOLO용으로 데이터 전처리

BASE1 = "./ai06-level1-project/"
BASE2 = "./additional_training_data/"
IMG_DIR1 = os.path.join(BASE1, "train_output")
IMG_DIR2 = os.path.join(BASE2, "train_cleaned")
ANN_FILE1 = os.path.join("./ai06-level1-project/", "train.json")
ANN_FILE2 = os.path.join("./additional_training_data/", "train.json")
TEST_IMG_DIR = os.path.join(BASE1, "test_images")

OUT_DIR = "yolo_dataset"

os.makedirs(os.path.join(OUT_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "images/val"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/val"), exist_ok=True)

with open(ANN_FILE1, "r", encoding="utf-8") as f:
    dataset1 = json.load(f)

with open(ANN_FILE2, "r", encoding="utf-8") as f:
    dataset2 = json.load(f)

i=pd.DataFrame(dataset1["images"] + dataset2["images"])
i=i.drop_duplicates()
i=i.to_dict(orient='records')

cat=pd.DataFrame(dataset1["categories"] + dataset2["categories"])
cat=cat.drop_duplicates()
cat=cat.to_dict(orient='records')

dataset = {
    "images": i,
    "annotations": dataset1["annotations"] + dataset2["annotations"],
    "categories": cat
}

dataset['categories']

[{'id': 31704, 'supercategory': 'pill', 'name': '낙소졸정 500/20mg'},
 {'id': 16550, 'supercategory': 'pill', 'name': '동아가바펜틴정 800mg'},
 {'id': 10223, 'supercategory': 'pill', 'name': '넥시움정 40mg'},
 {'id': 1899, 'supercategory': 'pill', 'name': '보령부스파정 5mg'},
 {'id': 33008, 'supercategory': 'pill', 'name': '신바로정'},
 {'id': 21025, 'supercategory': 'pill', 'name': '펠루비정(펠루비프로펜)'},
 {'id': 16547, 'supercategory': 'pill', 'name': '가바토파정 100mg'},
 {'id': 18109, 'supercategory': 'pill', 'name': '란스톤엘에프디티정 30mg'},
 {'id': 27925, 'supercategory': 'pill', 'name': '울트라셋이알서방정'},
 {'id': 29344, 'supercategory': 'pill', 'name': '비모보정 500/20mg'},
 {'id': 29450, 'supercategory': 'pill', 'name': '레일라정'},
 {'id': 19606, 'supercategory': 'pill', 'name': '스토가정 10mg'},
 {'id': 21770, 'supercategory': 'pill', 'name': '라비에트정 20mg'},
 {'id': 24849, 'supercategory': 'pill', 'name': '놀텍정 10mg'},
 {'id': 33207, 'supercategory': 'pill', 'name': '에스원엠프정 20mg'},
 {'id': 44198, 'supercategory': 'pill', 'name': '케이캡정 50

In [20]:
len(dataset['categories'])

74

잘 합쳐진듯 하니 이제 데이터셋 생성을 수행한다.

In [22]:
#YOLO용으로 데이터 전처리

OUT_DIR = "./yolo_dataset"

os.makedirs(os.path.join(OUT_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "images/val"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/val"), exist_ok=True)

coco = COCO()
coco.dataset = dataset
coco.createIndex()

img_ids = list(coco.imgs.keys())

train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)


def convert_to_yolo_bbox(box, img_w, img_h):
    x, y, w, h = box
    cx = (x + w/2) / img_w
    cy = (y + h/2) / img_h
    w /= img_w
    h /= img_h
    return cx, cy, w, h


def process_image(img_id, split="train"):

    img_info = coco.loadImgs(img_id)[0]
    file_name = img_info["file_name"]
    width, height = img_info["width"], img_info["height"]

    src_img_path1 = os.path.join(IMG_DIR1, file_name)
    src_img_path2 = os.path.join(IMG_DIR2, file_name)
    dst_img_path = os.path.join(OUT_DIR, f"images/{split}/{file_name}")

    if os.path.exists(src_img_path1):
        shutil.copy(src_img_path1, dst_img_path)
    elif os.path.exists(src_img_path2):
        shutil.copy(src_img_path2, dst_img_path)
    else:
        print("이미지 없음")
        return

    label_path = os.path.join(OUT_DIR, f"labels/{split}/{file_name.replace('.png', '.txt')}")

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    with open(label_path, "w", encoding="utf-8") as f:
        for ann in anns:
            category_id = ann["category_id"]
            yolo_class = list(coco.cats.keys()).index(category_id)
            bbox = ann["bbox"]
            yolo_box = convert_to_yolo_bbox(bbox, width, height)

            f.write(f"{yolo_class} {' '.join([str(round(v, 6)) for v in yolo_box])}\n")


for img_id in train_ids:
    process_image(img_id, split="train")

for img_id in val_ids:
    process_image(img_id, split="val")

print("YOLO dataset 생성 완료")


yaml_path = os.path.join(OUT_DIR, "data.yaml")
num_classes = len(coco.cats)
names = [coco.cats[k]["name"] for k in sorted(coco.cats.keys())]

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(f"path: {OUT_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")
    f.write(f"nc: {num_classes}\n")
    f.write(f"names: {names}\n")

print("data.yaml 파일 생성 완료")

creating index...
index created!
YOLO dataset 생성 완료
data.yaml 파일 생성 완료


# YOLO 학습하기
앞서 완성한 데이터셋을 활용하여 YOLO 모델을 학습한다.

In [25]:
model = YOLO("yolov10x.pt")

In [ ]:
model.train(
    data=r"./yolo_dataset/data.yaml",
    epochs=500,
    imgsz=640,
    batch=4,
    lr0=1e-5,
    lrf=0.2,
    device=0,
    workers=2,
    amp=True,
    name="pill_yolo10x",
    pretrained=True,
    seed=42,
    optimizer='Adam',
    project='./ai06-level1-project/train_checkpoints'
)

New https://pypi.org/project/ultralytics/8.3.237 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.235 🚀 Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10x.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=pill_yolo10x2, nb